In [1]:
import pandas as pd
import glob
import time
import duckdb
import sqlite3
import matplotlib

In [2]:
conn =  duckdb.connect('taxinyc.db')

In [3]:
df = conn.execute ("""
    CREATE TABLE IF NOT EXISTS trips AS
    SELECT * 
    FROM "../data/raw/yellow/yellow_tripdata_2025-01.parquet"
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [4]:
conn.execute("SELECT * FROM trips LIMIT 5").df()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1,1.60,1,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1,0.50,1,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1,0.60,1,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3,0.52,1,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3,0.66,1,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [10]:
conn.execute("SUMMARIZE trips").df()

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,7,4,1.7854283433652949,0.4263282213054444,2,2,2,3475226,0.00
1,tpep_pickup_datetime,TIMESTAMP,2024-12-31 20:47:55,2025-02-01 00:00:44,1967401,2025-01-17 11:02:55.910964,None,2025-01-10 10:16:13.81883,2025-01-17 15:55:49.016921,2025-01-24 18:49:53.154272,3475226,0.00
2,tpep_dropoff_datetime,TIMESTAMP,2024-12-18 07:52:40,2025-02-01 23:44:11,1773673,2025-01-17 11:17:56.997901,None,2025-01-10 08:59:57.401312,2025-01-17 16:33:47.455837,2025-01-24 19:48:57.993054,3475226,0.00
3,passenger_count,BIGINT,0,9,11,1.2978589658806226,0.750750275480472,1,1,1,3475226,15.54
4,trip_distance,DOUBLE,0.0,276423.57,3330,5.855126178844152,564.6015996346288,0.976511242890175,1.666747307505111,3.1071299984117027,3475226,0.00
5,RatecodeID,BIGINT,1,99,7,2.482534529758504,11.632772004033395,1,1,1,3475226,15.54
6,store_and_fwd_flag,VARCHAR,N,Y,2,None,None,None,None,None,3475226,15.54
7,PULocationID,INTEGER,1,265,266,165.1915757421244,64.52948262355439,132,162,234,3475226,0.00
8,DOLocationID,INTEGER,1,265,290,164.12517689497028,69.40168629828571,114,162,234,3475226,0.00
9,payment_type,BIGINT,0,5,6,1.0366229419324096,0.7013334099485083,1,1,1,3475226,0.00


In [9]:
conn.execute("""
    SELECT
        tpep_pickup_datetime.MIN() AS first_pickup,
        tpep_pickup_datetime.MAX() AS last_pickup,
        tpep_dropoff_datetime.MIN() AS first_dropoff,
        tpep_dropoff_datetime.MAX() AS last_dropoff,
        COUNT(*) AS total_trips
    FROM trips"""
).df()

,first_pickup,last_pickup,first_dropoff,last_dropoff,total_trips
0,2024-12-31 20:47:55,2025-02-01 00:00:44,2024-12-18 07:52:40,2025-02-01 23:44:11,3475226


In [13]:
conn.execute(
    """
    SELECT COUNT(*)
    FILTER (WHERE tpep_dropoff_datetime < tpep_pickup_datetime) AS dropoff_before_pickup
    FROM trips"""
).df()

,dropoff_before_pickup
0,124


In [14]:
conn.execute(
    """
    SELECT COUNT(*)
    FILTER (WHERE tpep_dropoff_datetime == tpep_pickup_datetime) AS dropoff_before_pickup
    FROM trips"""
).df()

,dropoff_before_pickup
0,1927


In [5]:
conn.execute(
    """
    SELECT COUNT(*)
    FILTER (WHERE tpep_dropoff_datetime <= tpep_pickup_datetime) AS dropoff_before_pickup
    FROM trips
    """
).df()

,dropoff_before_pickup
0,2051


In [16]:
conn.execute(
    """
    SELECT COUNT(*)
    FILTER (WHERE passenger_count == 0) AS no_passengers,
    FROM trips"""
).df()

,no_passengers
0,24656


In [24]:
conn.execute(
    """SELECT COALESCE(SUM(row_count - 1), 0) AS duplicate_rows
    FROM (
    SELECT COUNT(*) AS row_count
    FROM trips
    GROUP BY ALL
    HAVING COUNT(*) > 1
);"""
).df()

,duplicate_rows
0,3475225.0


In [25]:
conn.execute("""
    SELECT
        *,
        COUNT(*) AS occurrence_count
    FROM trips
    GROUP BY ALL
    HAVING COUNT(*) > 1
    ORDER BY occurrence_count DESC
""").df()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,occurrence_count


In [29]:
conn.execute(
"""
    CREATE OR REPLACE TABLE trips_flagged AS 
    
    WITH calculated AS (
        SELECT *,
        trip_distance >= 50 AS flag_extreme_dist,
        EPOCH(
            tpep_dropoff_datetime - tpep_pickup_datetime
        )/3600 AS duration_hours,
        EPOCH(
            tpep_dropoff_datetime - tpep_pickup_datetime
        )/60 AS duration_mins
        FROM trips
    ),
    with_speed AS (
        SELECT *,
        tip_amount / NULLIF(fare_amount, 0) AS tip_rate,
        fare_amount / NULLIF(trip_distance, 0) AS fare_per_mile,
        trip_distance / NULLIF(duration_hours, 0) AS speed_mph,
        EXTRACT('hour' FROM tpep_pickup_datetime::TIME) AS pickup_hour,
        dayname(tpep_pickup_datetime) AS pickup_dayofw
        FROM calculated
    )
    
    SELECT *,
    (
        speed_mph > 80
        OR speed_mph < 0
        OR speed_mph IS NULL
    ) AS flag_impossible_speed
    FROM with_speed;
"""
)

conn.execute(
    """
    SELECT
        SUM(flag_extreme_dist::INTEGER) AS extreme_distance_count,
        SUM(flag_impossible_speed::INTEGER) AS impossible_speed_count
        FROM trips_flagged;"""
).df()

,extreme_distance_count,impossible_speed_count
0,534.0,3033.0


In [30]:
conn.execute (""" 
COPY(
    SELECT *
    FROM trips_flagged
    WHERE tpep_dropoff_datetime >= tpep_pickup_datetime
    AND trip_distance >= 0
    AND fare_amount >= 0
    AND total_amount >= 0
    AND PULocationID IS NOT NULL
    )
    TO '../data/processed/clean_duck_2025-01.parquet' 
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

In [31]:
conn.execute (
    """ 
    CREATE OR REPLACE VIEW clean_yellow AS SELECT * FROM '../data/processed/clean_duck_2025-01.parquet'"""
)

In [54]:
#Hourly mart demand
conn.execute (
    """ 
    COPY (SELECT
        SUM(passenger_count) AS total_passengers,
        COUNT(*) AS total_trips,
        AVG(trip_distance) AS avg_trip_distance,
        SUM(total_amount) AS revenue,
        AVG(tip_rate) * 100 AS avg_tip_rate,
        AVG(duration_mins) AS avg_duration,
    FROM clean_yellow
    GROUP BY pickup_hour
    ORDER BY pickup_hour ASC)
    
    TO "../data/marts/mart_duck_hourly.parquet"
    (FORMAT PARQUET, COMPRESSION ZSTD)
    """
)

In [43]:
#Daily mart demand
conn.execute (
    """ 
    SELECT
        pickup_dayofw,
        SUM(passenger_count) AS total_passengers,
        COUNT(*) AS total_trips,
        AVG(trip_distance) AS avg_trip_distance,
        SUM(total_amount) AS revenue,
        AVG(tip_rate) * 100 AS avg_tip_rate,
        AVG(duration_mins) AS avg_duration,
    FROM clean_yellow
    GROUP BY pickup_dayofw 
    ORDER BY total_trips DESC
    """
).df()

,pickup_dayofw,total_passengers,total_trips,avg_trip_distance,revenue,avg_tip_rate,avg_duration
0,Thursday,642112.0,590000,5.971884,16086115.25,19.997138,15.783780
1,Friday,633778.0,562235,5.678457,15208344.45,19.458600,15.646378
2,Wednesday,604213.0,550321,5.309459,14728913.95,20.321588,15.193285
3,Saturday,556724.0,456636,4.780722,11545270.80,19.902870,14.317191
4,Tuesday,465709.0,437218,5.213843,11669784.76,20.860431,15.167001
5,Sunday,436821.0,383982,6.184170,10355598.40,18.437940,13.718678
6,Monday,387884.0,350271,4.190123,10739182.86,21.222447,14.930152


In [46]:
#Pickup zone demand
conn.execute (
    """ 
    SELECT
        PULocationID,
        SUM(passenger_count) AS total_passengers,
        COUNT(*) AS total_trips,
        AVG(trip_distance) AS avg_trip_distance,
        SUM(total_amount) AS revenue,
        AVG(tip_rate) * 100 AS avg_tip_rate,
        AVG(duration_mins) AS avg_duration,
    FROM clean_yellow
    GROUP BY PULocationID
    ORDER BY total_trips DESC
    LIMIT 10
    """
).df()

,PULocationID,total_passengers,total_trips,avg_trip_distance,revenue,avg_tip_rate,avg_duration
0,161,193538.0,164176,2.175179,3939425.78,21.481191,14.044264
1,237,186597.0,159919,1.676397,3184451.59,23.041833,11.180657
2,236,175686.0,151774,2.472736,3087172.05,22.230891,11.554487
3,132,194728.0,137498,15.357512,11053479.22,14.171003,38.110614
4,230,158024.0,120106,2.787468,3238210.60,21.868370,15.028123
5,186,141589.0,115455,3.103772,2772810.71,21.344187,14.810018
6,162,131521.0,114471,2.173723,2706823.79,24.074578,13.402166
7,142,130013.0,107335,2.700578,2335946.83,22.471089,12.258471
8,239,108875.0,93344,4.995836,1982280.76,22.118670,12.067598
9,163,112431.0,92881,2.272391,2256626.31,21.900812,13.939180


In [47]:
#Common Dropoff
conn.execute (
    """ 
    SELECT
        DOLocationID,
        SUM(passenger_count) AS total_passengers,
        COUNT(*) AS total_trips,
        AVG(trip_distance) AS avg_trip_distance,
        SUM(total_amount) AS revenue,
        AVG(tip_rate) * 100 AS avg_tip_rate,
        AVG(duration_mins) AS avg_duration,
    FROM clean_yellow
    GROUP BY DOLocationID
    ORDER BY total_trips DESC
    LIMIT 10
    """
).df()

,DOLocationID,total_passengers,total_trips,avg_trip_distance,revenue,avg_tip_rate,avg_duration
0,236,185072.0,157530,2.009049,3347899.94,22.939819,11.657577
1,237,168943.0,146079,3.763510,2941734.21,23.001950,11.127197
2,161,148147.0,126428,3.749582,2990218.36,21.061826,13.865421
3,230,134234.0,102537,4.872804,2922940.38,21.001313,16.784730
4,170,106682.0,96145,3.583196,2197987.50,21.009767,12.500983
5,142,111116.0,95652,4.834020,2143103.93,22.449304,12.557092
6,239,111661.0,94587,4.164884,2189440.82,22.834410,12.712603
7,162,102314.0,90227,4.525604,2108245.32,23.846992,12.821854
8,141,102452.0,89981,3.204771,1965151.25,21.769533,11.926702
9,68,96138.0,85178,3.158952,2052114.60,20.699315,13.478100


In [53]:
#Tip behavior 
conn.execute (
    """ 
    WITH binned_data AS (
    SELECT
        passenger_count,
        trip_distance,
        total_amount,
        tip_rate,
        duration_mins,
        
        CASE 
            WHEN trip_distance >= 0 AND trip_distance < 1 THEN '0-1 mile'
            WHEN trip_distance >= 1 AND trip_distance < 3 THEN '1-3 miles'
            WHEN trip_distance >= 3 AND trip_distance < 7 THEN '3-7 miles'
            WHEN trip_distance >= 7 AND trip_distance < 15 THEN '5-10 miles'
            WHEN trip_distance >= 15 THEN '15+ miles'
        END AS distance_group
    FROM clean_yellow)
    
    SELECT
        distance_group,
        SUM(passenger_count) AS total_passengers,
        COUNT(*) AS total_trips,
        AVG(trip_distance) AS avg_trip_distance,
        SUM(total_amount) AS revenue,
        AVG(tip_rate) * 100 AS avg_tip_rate,
        AVG(duration_mins) AS avg_duration
    FROM binned_data
    GROUP BY distance_group
    ORDER BY total_trips DESC
    """
).df()

,distance_group,total_passengers,total_trips,avg_trip_distance,revenue,avg_tip_rate,avg_duration
0,1-3 miles,1821164.0,1624207,1.748819,34130596.07,19.814630,12.481508
1,0-1 mile,954984.0,842544,0.613133,13506228.97,25.572149,6.853209
2,3-7 miles,505585.0,498499,4.345127,16093108.70,14.540388,21.551672
3,5-10 miles,260014.0,233803,9.812547,13823361.18,15.693726,30.670154
4,15+ miles,185494.0,131610,76.936328,12779915.55,15.208336,46.926281


In [12]:
conn.close()